In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, f1_score


In [ ]:
# Data preparation complete. Ready for model training.
#   prepared_full_v81.csv      — 54,136 rows × 80 cols
#   prepared_positive_v81.csv  — 20,354 rows (assigned only)
#   prepared_negative_v81.csv  — 33,782 rows (unassigned)
#   leads_full_v81.csv         — 11,474 rows (incl. re-entries)
#   brokers_full_v81.csv       — 306 rows (incl. replacements)
#   counterfactual_clean_v81.csv — 34,422 rows (clean)

In [4]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
DROPOUT = 0.3
LEARNING_RATE = 1e-3
BATCH_SIZE = 512
EPOCHS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FOCAL_ALPHA = 0.85
FOCAL_GAMMA = 3.0

In [5]:
print(f"Device: {DEVICE}")

Device: cuda


In [6]:
df = pd.read_csv("prepared_positive_v81.csv")
print(f"Loaded: {df.shape}")
print(f"Conversion rate: {df['converted'].mean()*100:.2f}%")

Loaded: (20354, 80)
Conversion rate: 11.68%


/tmp/ipykernel_73847/1236429721.py:1: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("prepared_positive_v81.csv")


In [7]:
CLIENT_FEATURES = [
    "quote_value", "lead_difficulty", "sophistication", "patience_hours",
    "digital_engagement_score", "tenure_years",
    "log_quote_value", "log_patience_hours",
    "month", "hour_of_day", "lead_dayofweek", "lead_quarter",
    "is_weekend", "insurance_type_enc", "claims_risk",
    "multi_product_intent", "insurance_type_missing", "language_missing",
    "tenure_years_missing", "digital_engagement_score_missing",
]

BROKER_FEATURES = [
    "skill_level", "conversion_rate", "csat_score", "reliability",
    "efficiency", "avg_response_time", "burnout_risk",
    "commission_rate", "cost_per_lead", "utilization",
    "ribo_licensed", "is_new_broker",
    "expertise_auto", "expertise_home", "expertise_bundle",
    "broker_quality_score",
    # one-hot encoded languages (created in preprocessing)
    "lang_Bilingual", "lang_English", "lang_French",
]

INTERACTION_FEATURES = [
    "expertise_match", "language_match", "workload_ratio", "quality_x_value",
    "position_bias", "interaction_number",
    "responded", "response_time_bucket_ord", "log_response_time_hours",
    "ribo_x_expertise", "claims_x_skill", "tenure_x_quality",
]

# Filter to columns that actually exist
def filter_cols(cols, df):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"  Warning — missing columns (skipped): {missing}")
    return [c for c in cols if c in df.columns]

CLIENT_FEATURES      = filter_cols(CLIENT_FEATURES, df)
BROKER_FEATURES      = filter_cols(BROKER_FEATURES, df)
INTERACTION_FEATURES = filter_cols(INTERACTION_FEATURES, df)

print(f"Client features:      {len(CLIENT_FEATURES)}")
print(f"Broker features:      {len(BROKER_FEATURES)}")
print(f"Interaction features: {len(INTERACTION_FEATURES)}")


Client features:      20
Broker features:      19
Interaction features: 12


In [8]:
all_feature_cols = CLIENT_FEATURES + BROKER_FEATURES + INTERACTION_FEATURES + ["converted"]
df = df[all_feature_cols].copy()


In [9]:
if "lead_year" in df.columns and "lead_quarter" in df.columns:
    df = df.sort_values(["lead_year", "lead_quarter"]).reset_index(drop=True)
    n = len(df)
    train_end = int(n * 0.70)
    val_end   = int(n * 0.85)
    train_df  = df.iloc[:train_end].copy()
    val_df    = df.iloc[train_end:val_end].copy()
    test_df   = df.iloc[val_end:].copy()
    print("Temporal split based on lead_year/lead_quarter")
else:
    print("Warning: lead_year/lead_quarter not found; using random split.")
    train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["converted"])
    val_df, test_df   = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["converted"])

print(f"Train: {len(train_df):,}  ({train_df['converted'].mean()*100:.1f}% positive)")
print(f"Val:   {len(val_df):,}  ({val_df['converted'].mean()*100:.1f}% positive)")
print(f"Test:  {len(test_df):,}  ({test_df['converted'].mean()*100:.1f}% positive)")

Train: 14,247  (11.7% positive)
Val:   3,053  (11.7% positive)
Test:  3,054  (11.7% positive)


In [10]:
all_features_to_scale = CLIENT_FEATURES + BROKER_FEATURES + INTERACTION_FEATURES

In [11]:
scaler = StandardScaler()

# Prepare train data: fill NaNs with column median (per column)
train_scaled_data = train_df[all_features_to_scale].copy()
for col in all_features_to_scale:
    if train_scaled_data[col].isna().any():
        median_val = train_scaled_data[col].median()
        train_scaled_data[col] = train_scaled_data[col].fillna(median_val)
        # Also store median for later use on val/test
        # We'll store it in a dict for later imputation
        if 'median_vals' not in locals():
            median_vals = {}
        median_vals[col] = median_val

# Fit scaler on train data
scaler.fit(train_scaled_data)


,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [12]:
def transform_with_scaler(df_part, scaler, median_vals, feature_cols):
    df_part_scaled = df_part[feature_cols].copy()
    for col in feature_cols:
        if df_part_scaled[col].isna().any():
            df_part_scaled[col] = df_part_scaled[col].fillna(median_vals.get(col, 0))
    scaled = scaler.transform(df_part_scaled)
    return scaled

train_scaled = transform_with_scaler(train_df, scaler, median_vals, all_features_to_scale)
val_scaled   = transform_with_scaler(val_df,   scaler, median_vals, all_features_to_scale)
test_scaled  = transform_with_scaler(test_df,  scaler, median_vals, all_features_to_scale)

# Create dataframes with scaled features (same column names, but we'll replace original columns)
for i, col in enumerate(all_features_to_scale):
    train_df[col] = train_scaled[:, i]
    val_df[col]   = val_scaled[:, i]
    test_df[col]  = test_scaled[:, i]

print("Features scaled (fit on train only).")


Features scaled (fit on train only).


In [13]:
class BrokerMatchDataset(Dataset):
    def __init__(self, df, client_cols, broker_cols, interaction_cols):
        self.client_x      = torch.tensor(df[client_cols].values, dtype=torch.float32)
        self.broker_x      = torch.tensor(df[broker_cols].values, dtype=torch.float32)
        self.interaction_x = torch.tensor(df[interaction_cols].values, dtype=torch.float32)
        self.labels        = torch.tensor(df["converted"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.client_x[idx],
            self.broker_x[idx],
            self.interaction_x[idx],
            self.labels[idx],
        )

train_ds = BrokerMatchDataset(train_df, CLIENT_FEATURES, BROKER_FEATURES, INTERACTION_FEATURES)
val_ds   = BrokerMatchDataset(val_df,   CLIENT_FEATURES, BROKER_FEATURES, INTERACTION_FEATURES)
test_ds  = BrokerMatchDataset(test_df,  CLIENT_FEATURES, BROKER_FEATURES, INTERACTION_FEATURES)

# WeightedRandomSampler for balanced batches
labels_array = train_df["converted"].values.astype(int)
class_counts = np.bincount(labels_array)
class_weights = 1.0 / class_counts
sample_weights = class_weights[labels_array]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float32),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,   num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,   num_workers=0)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")
print(f"Class counts — neg: {class_counts[0]:,}  pos: {class_counts[1]:,}  ratio: {class_counts[0]/class_counts[1]:.1f}x")

Train batches: 28
Val batches:   6
Test batches:  6
Class counts — neg: 12,583  pos: 1,664  ratio: 7.6x


In [14]:
class Tower(nn.Module):
    def __init__(self, input_dim, hidden_dim=HIDDEN_DIM, embed_dim=EMBEDDING_DIM, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 2, embed_dim),
        )

    def forward(self, x):
        emb = self.net(x)
        return nn.functional.normalize(emb, p=2, dim=1)

class TwoTowerModel(nn.Module):
    def __init__(self, client_dim, broker_dim, interaction_dim, embed_dim=EMBEDDING_DIM):
        super().__init__()
        self.client_tower = Tower(client_dim, embed_dim=embed_dim)
        self.broker_tower = Tower(broker_dim, embed_dim=embed_dim)

        fusion_input_dim = embed_dim + embed_dim + embed_dim + 1 + interaction_dim
        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_input_dim, 64),
            nn.ReLU(),
            nn.Dropout(DROPOUT * 0.5),
            nn.Linear(64, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, client_x, broker_x, interaction_x):
        client_emb = self.client_tower(client_x)
        broker_emb = self.broker_tower(broker_x)
        hadamard   = client_emb * broker_emb
        dot        = hadamard.sum(dim=1, keepdim=True)
        fusion_input = torch.cat([client_emb, broker_emb, hadamard, dot, interaction_x], dim=1)
        return self.fusion_head(fusion_input).squeeze(1)

model = TwoTowerModel(
    client_dim      = len(CLIENT_FEATURES),
    broker_dim      = len(BROKER_FEATURES),
    interaction_dim = len(INTERACTION_FEATURES),
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal trainable parameters: {total_params:,}")

TwoTowerModel(
  (client_tower): Tower(
    (net): Sequential(
      (0): Linear(in_features=20, out_features=128, bias=True)
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=128, out_features=64, bias=True)
      (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): Dropout(p=0.15, inplace=False)
      (8): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (broker_tower): Tower(
    (net): Sequential(
      (0): Linear(in_features=19, out_features=128, bias=True)
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=128, out_features=64, bias=True)
      (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
   

In [15]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, labels):
        bce = F.binary_cross_entropy_with_logits(logits, labels, reduction='none')
        pt = torch.exp(-bce)
        alpha_t = self.alpha * labels + (1 - self.alpha) * (1 - labels)
        focal = alpha_t * (1 - pt) ** self.gamma * bce
        return focal.mean()

loss_fn = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)

print(f"Focal Loss: alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA}")
print(f"Optimizer: AdamW  lr={LEARNING_RATE}  weight_decay=1e-4")

Focal Loss: alpha=0.85, gamma=3.0
Optimizer: AdamW  lr=0.001  weight_decay=1e-4


In [16]:
tiny_pos = train_df[train_df["converted"] == 1].head(50)
tiny_neg = train_df[train_df["converted"] == 0].head(50)
tiny_df  = pd.concat([tiny_pos, tiny_neg]).sample(frac=1, random_state=SEED).reset_index(drop=True)

tiny_ds = BrokerMatchDataset(tiny_df, CLIENT_FEATURES, BROKER_FEATURES, INTERACTION_FEATURES)
tiny_loader = DataLoader(tiny_ds, batch_size=16, shuffle=True)

_test_model = TwoTowerModel(
    client_dim      = len(CLIENT_FEATURES),
    broker_dim      = len(BROKER_FEATURES),
    interaction_dim = len(INTERACTION_FEATURES),
).to(DEVICE)
_test_loss_fn = FocalLoss()
_test_optimizer = torch.optim.AdamW(_test_model.parameters(), lr=1e-3)

print("Overfit test (50 pos + 50 neg samples):")
print(f"{'Epoch':>6}  {'Loss':>8}  {'AUC':>8}")

for ep in range(1, 51):
    _test_model.train()
    all_probs, all_labels_ovf = [], []
    for client_x, broker_x, inter_x, labels in tiny_loader:
        client_x, broker_x, inter_x, labels = (
            client_x.to(DEVICE), broker_x.to(DEVICE),
            inter_x.to(DEVICE), labels.to(DEVICE)
        )
        logits = _test_model(client_x, broker_x, inter_x)
        loss = _test_loss_fn(logits, labels)
        _test_optimizer.zero_grad()
        loss.backward()
        _test_optimizer.step()
        all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
        all_labels_ovf.extend(labels.cpu().numpy())
    if ep % 10 == 0:
        auc = roc_auc_score(all_labels_ovf, all_probs)
        print(f"{ep:>6}  {loss.item():>8.4f}  {auc:>8.4f}")

final_auc = roc_auc_score(all_labels_ovf, all_probs)
if final_auc > 0.85:
    print(f"\n✅ Sanity check PASSED (AUC={final_auc:.3f}) — model can learn, proceed to full training")
else:
    print(f"\n❌ Sanity check FAILED (AUC={final_auc:.3f}) — model has a fundamental issue, debug before training")

Overfit test (50 pos + 50 neg samples):
 Epoch      Loss       AUC
    10    0.0184    0.8400
    20    0.0125    0.9828
    30    0.0096    0.9816
    40    0.0042    0.9960
    50    0.0038    0.9976

✅ Sanity check PASSED (AUC=0.998) — model can learn, proceed to full training


In [17]:
def run_epoch(loader, train=True):
    model.train(train)
    total_loss, all_probs, all_labels = 0.0, [], []

    with torch.set_grad_enabled(train):
        for batch in loader:
            client_x, broker_x, inter_x, labels = batch
            client_x = client_x.to(DEVICE)
            broker_x = broker_x.to(DEVICE)
            inter_x  = inter_x.to(DEVICE)
            labels   = labels.to(DEVICE)

            logits = model(client_x, broker_x, inter_x)
            loss   = loss_fn(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * len(labels)
            all_probs.extend(torch.sigmoid(logits).cpu().detach().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(all_labels)
    auc      = roc_auc_score(all_labels, all_probs)
    pr_auc   = average_precision_score(all_labels, all_probs)
    return avg_loss, auc, pr_auc

best_val_auc = 0.0
best_epoch   = 0
history      = []

print(f"\n{'Epoch':>5}  {'Train Loss':>10}  {'Train AUC':>10}  {'Val Loss':>9}  {'Val AUC':>8}  {'Val PR-AUC':>10}")
print("-" * 65)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_auc, train_pr = run_epoch(train_loader, train=True)
    val_loss,   val_auc,   val_pr   = run_epoch(val_loader,   train=False)

    scheduler.step(val_auc)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss, "train_auc": train_auc, "train_pr": train_pr,
        "val_loss":   val_loss,   "val_auc":   val_auc,   "val_pr":   val_pr,
    })

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch   = epoch
        torch.save(model.state_dict(), "two_tower_best.pt")

    print(
        f"{epoch:>5}  {train_loss:>10.4f}  {train_auc:>10.4f}  "
        f"{val_loss:>9.4f}  {val_auc:>8.4f}  {val_pr:>10.4f}"
        + ("  ← best" if epoch == best_epoch else "")
    )

print(f"\nBest model: epoch {best_epoch}  |  val AUC = {best_val_auc:.4f}")


Epoch  Train Loss   Train AUC   Val Loss   Val AUC  Val PR-AUC
-----------------------------------------------------------------
    1      0.0354      0.5316     0.0386    0.6728      0.1948  ← best
    2      0.0264      0.6632     0.0293    0.6882      0.1972  ← best
    3      0.0250      0.7009     0.0293    0.6887      0.2011  ← best
    4      0.0246      0.7134     0.0268    0.6890      0.2038  ← best
    5      0.0240      0.7259     0.0289    0.6950      0.2043  ← best
    6      0.0237      0.7307     0.0263    0.6969      0.2046  ← best
    7      0.0236      0.7349     0.0270    0.6937      0.2080
    8      0.0237      0.7279     0.0285    0.6925      0.2049
    9      0.0234      0.7369     0.0282    0.6955      0.2120
   10      0.0231      0.7424     0.0265    0.6965      0.2100
   11      0.0230      0.7458     0.0262    0.6930      0.2014
   12      0.0225      0.7631     0.0280    0.6892      0.2030
   13      0.0224      0.7633     0.0271    0.6920      0.1995
   

In [18]:
model.load_state_dict(torch.load("two_tower_best.pt", map_location=DEVICE))
model.eval()

val_probs, val_labels_list = [], []
with torch.no_grad():
    for client_x, broker_x, inter_x, labels in val_loader:
        logits = model(client_x.to(DEVICE), broker_x.to(DEVICE), inter_x.to(DEVICE))
        val_probs.extend(torch.sigmoid(logits).cpu().numpy())
        val_labels_list.extend(labels.numpy())

val_probs_arr  = np.array(val_probs)
val_labels_arr = np.array(val_labels_list)

thresholds = np.arange(0.05, 0.90, 0.01)
best_t, best_f1 = 0.5, 0.0
results = []
for t in thresholds:
    preds = (val_probs_arr >= t).astype(int)
    f1    = f1_score(val_labels_arr, preds, zero_division=0)
    results.append((t, f1))
    if f1 > best_f1:
        best_f1, best_t = f1, t

print(f"Optimal threshold: {best_t:.2f}  (val F1 for 'convert' = {best_f1:.3f})")
print()
print(f"{'Threshold':>10}  {'F1':>8}")
for t, f1 in results:
    if abs(t - best_t) < 0.08:
        marker = "  ← optimal" if t == best_t else ""
        print(f"{t:>10.2f}  {f1:>8.4f}{marker}")


Optimal threshold: 0.63  (val F1 for 'convert' = 0.290)

 Threshold        F1
      0.55    0.2746
      0.56    0.2780
      0.57    0.2778
      0.58    0.2820
      0.59    0.2840
      0.60    0.2817
      0.61    0.2862
      0.62    0.2875
      0.63    0.2903  ← optimal
      0.64    0.2895
      0.65    0.2785
      0.66    0.2736
      0.67    0.2426
      0.68    0.2034
      0.69    0.1434
      0.70    0.1062


In [19]:
test_loss, test_auc, test_pr_auc = run_epoch(test_loader, train=False)

all_probs, all_labels_test = [], []
with torch.no_grad():
    for client_x, broker_x, inter_x, labels in test_loader:
        logits = model(client_x.to(DEVICE), broker_x.to(DEVICE), inter_x.to(DEVICE))
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels_test.extend(labels.numpy())

all_probs_arr  = np.array(all_probs)
all_labels_arr = np.array(all_labels_test)

all_preds_tuned  = (all_probs_arr >= best_t).astype(int)
all_preds_half   = (all_probs_arr >= 0.50).astype(int)

print("=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(f"  Loss:              {test_loss:.4f}")
print(f"  AUC-ROC:           {test_auc:.4f}")
print(f"  PR-AUC:            {test_pr_auc:.4f}")
print(f"  Threshold used:    {best_t:.2f}  (tuned on val)")
print()
print(f"--- With tuned threshold ({best_t:.2f}) ---")
print(classification_report(all_labels_arr, all_preds_tuned, target_names=["no convert", "convert"]))
print(f"--- With default threshold (0.50) for comparison ---")
print(classification_report(all_labels_arr, all_preds_half,  target_names=["no convert", "convert"]))

pos_mask = all_labels_arr == 1
neg_mask = all_labels_arr == 0
print("Probability distribution:")
print(f"  Mean prob for positives: {all_probs_arr[pos_mask].mean():.3f}")
print(f"  Mean prob for negatives: {all_probs_arr[neg_mask].mean():.3f}")
print(f"  (These should be clearly different. If similar, model isn't discriminating.)")


TEST SET RESULTS
  Loss:              0.0283
  AUC-ROC:           0.6843
  PR-AUC:            0.1946
  Threshold used:    0.63  (tuned on val)

--- With tuned threshold (0.63) ---
              precision    recall  f1-score   support

  no convert       0.92      0.68      0.78      2697
     convert       0.19      0.57      0.28       357

    accuracy                           0.66      3054
   macro avg       0.56      0.62      0.53      3054
weighted avg       0.84      0.66      0.72      3054

--- With default threshold (0.50) for comparison ---
              precision    recall  f1-score   support

  no convert       0.95      0.36      0.52      2697
     convert       0.15      0.87      0.26       357

    accuracy                           0.42      3054
   macro avg       0.55      0.61      0.39      3054
weighted avg       0.86      0.42      0.49      3054

Probability distribution:
  Mean prob for positives: 0.606
  Mean prob for negatives: 0.474
  (These should be cl